In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

# Previsão direta somente pela LLM: preços conhecidos + manchetes

A LLM recebe 20 retornos diários conhecidos do ativo, os retornos conhecidos dos oito ativos na data de decisão, volatilidade passada e até oito manchetes verificadas disponíveis nas cinco sessões anteriores. Ela devolve diretamente uma previsão numérica do retorno relativo dos próximos cinco pregões e a evidência textual. **Não há embeddings MiniLM ou MOMENT, modelo auxiliar treinado nem inferência no computador local.** Respostas e prompts são auditáveis e retomáveis.

O teste 2022–23 já foi inspecionado; serve como comparação exploratória, não confirmação de alfa. Modelos treinados após esse período podem ter memorizado resultados. Não há avaliação 2024–25: o feed histórico completo ainda não está disponível. Não use textos dos corpos dos artigos. Compara-se com previsão de excesso zero, baseline numérica previamente salva e comprar e manter dos mesmos oito ativos, após custos.

O backend `api` exige endpoint de **chat completions compatível com OpenAI** e identificador real fornecido pelo provedor; o nome comercial ‘GPT 6 Luna’ não foi verificado como identificador de API. O backend `transformers` executa um modelo instruct em 4 bits na GPU remota; 25 GB podem bastar para modelos de ~7–8 bilhões de parâmetros, mas memória de contexto e compatibilidade dependem dos pesos escolhidos. Não execute os modelos localmente.

In [ ]:
import importlib.util
import subprocess
import hashlib

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Execute este notebook no Google Colab, não no computador local') from exc
drive.mount('/content/drive')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO))
DATA = Path('/content/drive/MyDrive/latent_fusion_data')
PANEL = DATA / 'fusion_event_panel.npz'
NEWS = DATA / 'fusion_events.jsonl.gz'
BASELINE = DATA / 'fast_fusion_demo_results' / 'predictions_ts_only.csv'
for path in (PANEL, NEWS, BASELINE):
    if not path.is_file():
        raise FileNotFoundError(f'Artefato auditado necessário: {path}')
{'painel': str(PANEL), 'manchetes': str(NEWS), 'baseline': str(BASELINE)}

## Configurar provedor antes de inferir

Defina `LLM_CONTEXT_BACKEND=api` e `LLM_CONTEXT_MODEL_ID` com o identificador **exato** habilitado na sua conta para testar o serviço hospedado. Se necessário, defina `LLM_CONTEXT_BASE_URL` com a URL de API compatível com chat completions; a chave é solicitada sem aparecer na saída. Para modelo aberto, defina `LLM_CONTEXT_BACKEND=transformers` e `LLM_CONTEXT_MODEL_ID=Qwen/Qwen2.5-7B-Instruct`, ou outro modelo instruct compatível com Transformers e sua GPU. O notebook não atribui um nome de API não confirmado ao modelo comercial. Cada provedor/modelo escreve em pasta própria no Drive.

Exemplo em uma célula privada do Colab, antes da célula seguinte: `os.environ['LLM_CONTEXT_MODEL_ID'] = 'Qwen/Qwen2.5-7B-Instruct'` e `os.environ['LLM_CONTEXT_BACKEND'] = 'transformers'`. Não publique tokens no notebook.

In [ ]:
import getpass
BACKEND = os.environ.get('LLM_CONTEXT_BACKEND', 'api')
MODEL_ID = os.environ.get('LLM_CONTEXT_MODEL_ID')
BASE_URL = os.environ.get('LLM_CONTEXT_BASE_URL') or None
if BACKEND not in ('api', 'transformers') or not MODEL_ID:
    raise ValueError('Defina LLM_CONTEXT_BACKEND e LLM_CONTEXT_MODEL_ID antes de inferir')
if BACKEND == 'api':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai>=2.0'], check=True)
    if not os.environ.get('OPENAI_API_KEY'):
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Chave da API (não será salva): ')
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Ative uma GPU remota para o modelo aberto')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.44,<5', 'accelerate', 'bitsandbytes'], check=True)
OUT = DATA / 'llm_context_results' / (BACKEND + '-' + hashlib.sha256(f'{MODEL_ID}|{BASE_URL}'.encode()).hexdigest()[:12])
{'backend': BACKEND, 'modelo': MODEL_ID, 'gpu': torch.cuda.get_device_name(0) if BACKEND == 'transformers' else 'API hospedada', 'saída': str(OUT)}

## Conferir contexto antes de enviar à LLM

A prévia abaixo não executa modelo. Cada decisão ocorre depois do fechamento: dados de preços vão somente até a data indicada; as manchetes já têm atraso conservador na data de disponibilidade. Campos de rótulos e retornos futuros ficam **fora** do prompt. Uma notícia ausente do feed após 2023 não é tratada como silêncio do mercado.

In [ ]:
from IPython.display import display
from src.fusion.llm_context_experiment import prepare_requests
panel, requests = prepare_requests(PANEL, NEWS)
display({'n_decisões': len(requests), 'exemplo_de_contexto': requests[0]['context']})

## Inferir, retomar e comparar

A LLM responde para **todas** as decisões do teste, inclusive sem manchete, usando só preços conhecidos nesse caso. Interrupções preservam respostas completas no JSONL; uma saída inválida para a execução antes de produzir métricas. Cache e resultados separados por modelo permitem comparações posteriores sem misturar respostas. Os pesos de carteira são 100% comprados, com custos de 10 bps, rebalanceamento a cada cinco sessões e execução após o fechamento de decisão. Não interprete uma melhora no teste já visto como validação independente.

In [ ]:
from IPython.display import Image
from src.fusion.llm_context_experiment import run_experiment
result = run_experiment(PANEL, NEWS, OUT, backend=BACKEND, model_id=MODEL_ID,
                        base_url=BASE_URL, baseline_csv=BASELINE, cost_bps=10)
display(result['signal'].round(6), result['portfolio'].round(4))
display(Image(filename=str(OUT / 'equity_llm_vs_bh.png')))
{'respostas_auditadas': str(result['cache']), 'métricas': str(OUT / 'portfolio_metrics.csv')}